In [109]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import math
import itertools

from seq_utils import generate_kv_mapping

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Constant

In [110]:
seq_folder = "/Users/ccnlab/Development/sequences/shaping/trans_mist"

NUM_KEYS = 3
NUM_FOOD = 3
NUM_VILLAGERS = 3
ALL_VF_IMAGES_SET = 3
TRANS_FOLDER = 1
OUTPUT_COL_ORDER = [
    "stim",
    "correct_key",
    "correct_food",
    "block",
    "img_folder",
    "trans_folder",
    "key0_trans",
    "key1_trans",
    "key2_trans",
    "shop0_food",
    "shop1_food",
    "shop2_food",
    "trans0_shop",
    "trans1_shop",
    "trans2_shop",
    "set_size",
]

SELF_PACED_OUTPUT_COL_ORDER = [
    "stim",
    "correct_key",
    "img_folder",
    "set_size",
    "type",
    "trans_folder",
]

In [111]:
# Get all unique permutations (not just combinations) of [0, 1, 2]
unique_food_layouts = np.array(list(itertools.permutations(np.arange(NUM_FOOD))))
print(unique_food_layouts)
unique_order = np.array(list(itertools.permutations(np.arange(len(unique_food_layouts)))))
print(np.array(unique_order).shape)

# valid_food_layouts_seq includes sequences that at least one food item is fixed
valid_food_layouts_seq = []
for order_list in unique_order:
    combination = [unique_food_layouts[o] for o in order_list]
    valid = True
    for i in range(len(combination)-1):
        if not np.any(combination[i] == combination[i+1]):
            valid = False
            break
    
    if valid:
        valid_food_layouts_seq.append(combination)

valid_food_layouts_seq = np.array(valid_food_layouts_seq)
print(valid_food_layouts_seq.shape)

[[0 1 2]
 [0 2 1]
 [1 0 2]
 [1 2 0]
 [2 0 1]
 [2 1 0]]
(720, 6)
(72, 6, 3)


In [112]:
trans_shop_mapping = generate_kv_mapping(NUM_KEYS, NUM_KEYS)
stim_food_mapping = generate_kv_mapping(NUM_VILLAGERS, NUM_FOOD)

STIM_ITER = 4
unique_stim_combinations = np.array(
    list(itertools.permutations(np.arange(NUM_VILLAGERS)))
)
# Randomly draw four combos from unique_combinations
unique_stim_seq = unique_stim_combinations[
    np.random.choice(len(unique_stim_combinations), size=STIM_ITER, replace=True)
]
unique_stim_seq = np.array(unique_stim_seq).flatten()
# (TODO) make sure there is no consecutive same stim
print("Stim (villager) sequence:")
print(unique_stim_seq)

food_layout_seq = valid_food_layouts_seq[
    np.random.choice(
        len(valid_food_layouts_seq), size=len(unique_stim_seq), replace=True
    )
]
# To flatten the first two dimensions in food_combination_seq:
food_layout_seq_flat = food_layout_seq.reshape(-1, *food_layout_seq.shape[2:])
print("Food layout sequence shape:", food_layout_seq_flat.shape)

unique_trans_combinations = np.array(list(itertools.permutations(np.arange(NUM_KEYS))))
trans_seq = unique_trans_combinations[
    np.random.choice(len(unique_trans_combinations), size=food_layout_seq_flat.shape[0], replace=True)
]
print("Trans sequence shape:", trans_seq.shape)

Stim (villager) sequence:
[0 2 1 0 1 2 2 1 0 0 1 2]
Food layout sequence shape: (72, 3)
Trans sequence shape: (72, 3)


In [114]:
from seq_utils import generate_seq_pair, shuffle_with_mask, swap_by_indices


def generate_non_shaping_block(
    num_keys,
    food_layout_seq_flat,
    num_iter_per_stim,
    trans_shop_mapping,
    stim_food_mapping,
):
    num_villagers = len(stim_food_mapping)
    num_trans = len(trans_shop_mapping)
    num_trials = food_layout_seq_flat.shape[0]

    interleaved_villager_seq, _ = generate_seq_pair(
        num_villagers, num_iter_per_stim, num_keys
    )
    villager_seq = np.repeat(
        interleaved_villager_seq, int(num_trials / len(interleaved_villager_seq))
    )

    _, correct_key_seq = generate_seq_pair(
        num_villagers, int(num_trials / num_villagers), num_keys
    )

    trans_seq = []
    for i in range(math.ceil(len(correct_key_seq) / num_trans)):
        trans_seq.extend(np.random.permutation(num_trans))

    seq_data = {
        "stim": villager_seq,
        "correct_key": correct_key_seq,
        "correct_food": [],
    }
    for i in range(num_keys):
        seq_data[f"shop{i}_food"] = []
        seq_data[f"key{i}_trans"] = []

    all_trans_indexes = np.arange(num_trans)
    shop_trans_mapping = np.argsort(trans_shop_mapping)
    for i, correct_key in enumerate(correct_key_seq):
        food_layout = food_layout_seq_flat[i]
        correct_food = stim_food_mapping[villager_seq[i]]
        seq_data["correct_food"].append(correct_food)

        correct_shop = np.where(food_layout == correct_food)[0][0]
        correct_trans = shop_trans_mapping[correct_shop]
        base = swap_by_indices(all_trans_indexes, correct_trans, correct_key)
        #print(base, correct_key, correct_trans)
        key_trans_array = shuffle_with_mask(
            base, np.array([i == correct_key for i in range(num_keys)])
        )

        for j, trans in enumerate(key_trans_array):
            seq_data[f"key{j}_trans"].append(trans)

        for j, food in enumerate(food_layout):
            seq_data[f"shop{j}_food"].append(food)

    for k, v in enumerate(trans_shop_mapping):
        seq_data[f"trans{k}_shop"] = [v] * len(villager_seq)

    return pd.DataFrame(seq_data)

In [115]:
nonshaping_block_seq = generate_non_shaping_block(
    NUM_KEYS, food_layout_seq_flat, STIM_ITER, trans_shop_mapping, stim_food_mapping
)

In [116]:
nonshaping_block_seq["block"] =  1
nonshaping_block_seq["img_folder"] = 1
nonshaping_block_seq["trans_folder"] = 1
nonshaping_block_seq["set_size"] = NUM_VILLAGERS
nonshaping_block_seq[OUTPUT_COL_ORDER].to_csv(
    f"{seq_folder}/nonshaping_trans0_learning.csv", index=False
)